# XGBoost & LightGBM — Classification de Sentiment en Fongbé

Ce notebook entraîne et compare **XGBoost** et **LightGBM** avec **class weighting natif** sur le corpus Fongbé.

**Pipeline :**
1. Vectorisation TF-IDF `char_wb` (3,5)
2. Class weights calculés automatiquement depuis la distribution
3. Optimisation Optuna (30 trials chacun) — **Objectif : Maximiser l'Accuracy**
4. Comparaison finale sur le jeu de test (15%)

| Label | Sentiment |
|-------|-----------|
| 0 | Neutre |
| 1 | Négatif |
| 2 | Positif |

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)
from sklearn.utils.class_weight import compute_sample_weight

import xgboost as xgb
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

print(f"XGBoost  : {xgb.__version__}")
print(f"LightGBM : {lgb.__version__}")
print("Imports OK.")

## 2. Chargement et préparation des données

In [ ]:
df = pd.read_csv("../data/corpus_final.csv", sep="|")
df["sentiment_final"] = pd.to_numeric(df["sentiment_final"], errors="coerce")
df = df.dropna(subset=["fon", "sentiment_final"])
df["label"] = df["sentiment_final"].astype(int)

LABEL_NAMES = {0: "Neutre", 1: "Négatif", 2: "Positif"}

print(f"Phrases valides : {len(df):,}")
print("\nDistribution :")
for lbl, cnt in df["label"].value_counts().sort_index().items():
    pct = cnt / len(df) * 100
    print(f"  {LABEL_NAMES[lbl]:>8s} ({lbl}) : {cnt:>6,} ({pct:.1f}%)")

## 3. Partitionnement (Train 70% / Val 15% / Test 15%)

Même `random_state=42` que le notebook original pour comparaison exacte.

In [ ]:
X = np.array(df["fon"].astype(str).tolist(), dtype=object)
y = np.array(df["label"].tolist(), dtype=int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train : {len(X_train):,}  |  Val : {len(X_val):,}  |  Test : {len(X_test):,}")
print(f"Total : {len(X):,}")

## 4. Vectorisation TF-IDF

In [ ]:
tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=40_000,
    sublinear_tf=True,
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print(f"Matrice TF-IDF Train : {X_train_tfidf.shape}")
print(f"Matrice TF-IDF Val   : {X_val_tfidf.shape}")
print(f"Matrice TF-IDF Test  : {X_test_tfidf.shape}")

## 5. Calcul des Class Weights

In [ ]:
class_counts = np.bincount(y_train)
n_classes = len(class_counts)
n_total = len(y_train)
class_weights = n_total / (n_classes * class_counts)

weight_dict = {i: w for i, w in enumerate(class_weights)}
sample_weights_train = np.array([class_weights[label] for label in y_train])

print("Class weights (balanced) :")
for i, w in weight_dict.items():
    print(f"  {LABEL_NAMES[i]:>8s} ({i}) : {w:.4f}  (n={class_counts[i]:,})")

## 6. XGBoost — Optimisation Optuna (Objectif : Accuracy)

In [ ]:
def xgb_objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
        'max_depth':       trial.suggest_int('max_depth', 4, 12),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':       trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':       trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':      trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':           trial.suggest_float('gamma', 0.0, 5.0),
    }

    model = xgb.XGBClassifier(
        **params,
        objective='multi:softprob',
        num_class=3,
        tree_method='hist',
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )
    model.fit(X_train_tfidf, y_train, sample_weight=sample_weights_train)
    preds = model.predict(X_val_tfidf)
    return accuracy_score(y_val, preds)


print("Optimisation XGBoost (30 trials)...")
xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_objective, n_trials=30, show_progress_bar=True)

print(f"\nMeilleure Accuracy (Val) : {xgb_study.best_value:.4f}")
print(f"Meilleurs hyperparamètres :")
for k, v in xgb_study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
best_xgb = xgb.XGBClassifier(
    **xgb_study.best_params,
    objective='multi:softprob',
    num_class=3,
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
best_xgb.fit(X_train_tfidf, y_train, sample_weight=sample_weights_train)

pred_val_xgb = best_xgb.predict(X_val_tfidf)
print("XGBoost — Validation :")
print(f"  Accuracy : {accuracy_score(y_val, pred_val_xgb):.4f}")
print(f"  F1 Macro : {f1_score(y_val, pred_val_xgb, average='macro'):.4f}")
print()
print(classification_report(y_val, pred_val_xgb, target_names=["Neutre", "Négatif", "Positif"], digits=4))

## 7. LightGBM — Optimisation Optuna (Objectif : Accuracy)

In [ ]:
def lgb_objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
        'max_depth':       trial.suggest_int('max_depth', 4, 15),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':      trial.suggest_int('num_leaves', 20, 150),
        'subsample':       trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':       trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':      trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
    }

    model = lgb.LGBMClassifier(
        **params,
        objective='multiclass',
        num_class=3,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_val_tfidf)
    return accuracy_score(y_val, preds)


print("Optimisation LightGBM (30 trials)...")
lgb_study = optuna.create_study(direction='maximize')
lgb_study.optimize(lgb_objective, n_trials=30, show_progress_bar=True)

print(f"\nMeilleure Accuracy (Val) : {lgb_study.best_value:.4f}")
print(f"Meilleurs hyperparamètres :")
for k, v in lgb_study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
best_lgbm = lgb.LGBMClassifier(
    **lgb_study.best_params,
    objective='multiclass',
    num_class=3,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
best_lgbm.fit(X_train_tfidf, y_train)

pred_val_lgbm = best_lgbm.predict(X_val_tfidf)
print("LightGBM — Validation :")
print(f"  Accuracy : {accuracy_score(y_val, pred_val_lgbm):.4f}")
print(f"  F1 Macro : {f1_score(y_val, pred_val_lgbm, average='macro'):.4f}")
print()
print(classification_report(y_val, pred_val_lgbm, target_names=["Neutre", "Négatif", "Positif"], digits=4))

## 8. Comparaison sur le Jeu de Test (15%)

In [ ]:
pred_test_xgb  = best_xgb.predict(X_test_tfidf)
pred_test_lgbm = best_lgbm.predict(X_test_tfidf)

try:
    tfidf_old   = joblib.load("../models/tfidf.joblib")
    voting_old  = joblib.load("../models/voting_model.joblib")
    X_test_old  = tfidf_old.transform(X_test)
    pred_test_voting = voting_old.predict(X_test_old)
    has_voting = True
except Exception as e:
    has_voting = False
    print("Ancien modèle Soft Voting non trouvé, comparaison limitée à XGBoost et LightGBM.")

model_preds = []
if has_voting:
    model_preds.append(("Soft Voting (existant)", pred_test_voting))
model_preds.append(("XGBoost + Class Weights", pred_test_xgb))
model_preds.append(("LightGBM + Class Weights", pred_test_lgbm))

results = []
for name, preds in model_preds:
    results.append({
        "Modèle": name,
        "Accuracy": accuracy_score(y_test, preds),
        "F1 Macro": f1_score(y_test, preds, average="macro"),
        "F1 Weighted": f1_score(y_test, preds, average="weighted"),
        "F1 Neutre": f1_score(y_test, preds, average=None)[0],
        "F1 Négatif": f1_score(y_test, preds, average=None)[1],
        "F1 Positif": f1_score(y_test, preds, average=None)[2],
    })

df_results = pd.DataFrame(results)
print("=" * 90)
print("COMPARAISON FINALE SUR LE JEU DE TEST")
print("=" * 90)
print(df_results.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
for name, preds in [("XGBoost + Class Weights", pred_test_xgb), ("LightGBM + Class Weights", pred_test_lgbm)]:
    print(f"\n{'=' * 60}")
    print(f"RAPPORT DÉTAILLÉ — {name}")
    print(f"{'=' * 60}")
    print(classification_report(y_test, preds, target_names=["Neutre", "Négatif", "Positif"], digits=4))

## 9. Matrices de Confusion

In [ ]:
fig, axes = plt.subplots(1, len(model_preds), figsize=(6 * len(model_preds), 5))
if len(model_preds) == 1:
    axes = [axes]

for ax, (name, preds) in zip(axes, model_preds):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Neutre", "Négatif", "Positif"],
                yticklabels=["Neutre", "Négatif", "Positif"])
    acc = accuracy_score(y_test, preds)
    f1m = f1_score(y_test, preds, average='macro')
    ax.set_title(f"{name}\nAccuracy={acc:.2%} | F1M={f1m:.2%}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Prédiction")
    ax.set_ylabel("Classe réelle")

plt.suptitle("Matrices de Confusion — Test Set", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 10. Ré-entraînement Final (Train+Val = 85%) et Sauvegarde du Meilleur Modèle (Sélection sur Accuracy)

In [ ]:
# Sélection du meilleur modèle basée sur l'ACCURACY sur le jeu de test
acc_xgb  = accuracy_score(y_test, pred_test_xgb)
acc_lgbm = accuracy_score(y_test, pred_test_lgbm)

if acc_xgb >= acc_lgbm:
    best_name = "XGBoost"
    best_params = xgb_study.best_params
    print(f"Meilleur modèle sélectionné (sur Accuracy) : XGBoost (Accuracy Test = {acc_xgb:.4f})")
else:
    best_name = "LightGBM"
    best_params = lgb_study.best_params
    print(f"Meilleur modèle sélectionné (sur Accuracy) : LightGBM (Accuracy Test = {acc_lgbm:.4f})")

X_train_val = np.concatenate([X_train, X_val])
y_train_val = np.concatenate([y_train, y_val])

tfidf_final = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=40_000,
    sublinear_tf=True,
    min_df=2
)
X_tv_tfidf = tfidf_final.fit_transform(X_train_val)
X_te_tfidf = tfidf_final.transform(X_test)

counts_tv = np.bincount(y_train_val)
cw_tv = len(y_train_val) / (len(counts_tv) * counts_tv)
sw_tv = np.array([cw_tv[l] for l in y_train_val])

if best_name == "XGBoost":
    final_model = xgb.XGBClassifier(
        **best_params,
        objective='multi:softprob',
        num_class=3,
        tree_method='hist',
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )
    final_model.fit(X_tv_tfidf, y_train_val, sample_weight=sw_tv)
else:
    final_model = lgb.LGBMClassifier(
        **best_params,
        objective='multiclass',
        num_class=3,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )
    final_model.fit(X_tv_tfidf, y_train_val)

pred_final = final_model.predict(X_te_tfidf)
final_acc = accuracy_score(y_test, pred_final)
final_f1m = f1_score(y_test, pred_final, average='macro')
final_f1w = f1_score(y_test, pred_final, average='weighted')

print(f"\n{'=' * 60}")
print(f"RÉSULTAT FINAL — {best_name} (Train+Val 85% → Test 15%)")
print(f"{'=' * 60}")
print(f"Accuracy       : {final_acc:.4f}")
print(f"F1-Score Macro  : {final_f1m:.4f}")
print(f"F1-Score Weighted: {final_f1w:.4f}")
print()
print(classification_report(y_test, pred_final, target_names=["Neutre", "Négatif", "Positif"], digits=4))

In [ ]:
os.makedirs("../models", exist_ok=True)

model_filename = f"{best_name.lower()}_model.joblib"
joblib.dump(tfidf_final, "../models/tfidf_v2.joblib")
joblib.dump(final_model, f"../models/{model_filename}")

print(f"Fichiers sauvegardés :")
print(f"  ../models/tfidf_v2.joblib   (vectorizer TF-IDF v2)")
print(f"  ../models/{model_filename}  ({best_name} final)")

## 11. Graphique Comparatif Final

In [ ]:
models_names = [m[0].replace(' ', '\n') for m in model_preds]
metrics_acc = [df_results.iloc[i]["Accuracy"] for i in range(len(model_preds))]
metrics_f1m = [df_results.iloc[i]["F1 Macro"] for i in range(len(model_preds))]

x = np.arange(len(models_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, metrics_acc, width, label='Accuracy', color='#4C9AFF', edgecolor='white')
bars2 = ax.bar(x + width/2, metrics_f1m, width, label='F1 Macro', color='#FF6B6B', edgecolor='white')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1%}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 5), textcoords="offset points",
                    ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparaison des Modèles (Optimisés sur Accuracy) — Test Set', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models_names, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()